# Wardrobe-AI virtual try-on — free GPU on Kaggle

Runs **the same model** as the rest of the app (FASHN VTON v1.5,
Apache-2.0) on Kaggle's free GPU, and prints a public address your
Wardrobe-AI backend can call.

## Why Kaggle rather than the Hugging Face Space

The public Space runs on Hugging Face **ZeroGPU**, whose free allowance
is roughly **2 GPU-minutes a day** for an anonymous caller and about
**3.5–5 minutes** for a signed-in free account. That allowance belongs
to the *caller*, so every user of your app shares the one allowance your
backend holds — which is a handful of images **per day for the whole
application**, not per user.

Kaggle gives a verified free account roughly **30 GPU-hours a week** on
a P100 or two T4s (16 GB), in sessions of up to 12 hours. The model needs
about 8 GB, so it fits comfortably. At roughly 30 seconds a try-on,
30 hours is on the order of **3,500 generations a week** — far more than
ten a day for every user you have.

The catch is uptime, not GPU time: the notebook has to be **running**.
It stops at the 12-hour session limit, when the browser tab is closed
for too long, or when Kaggle is busy. So this is the right thing to
start before a demo or a marking session, not something that stays up by
itself.

## Before you start

1. **Settings → Accelerator → GPU P100** (or **T4 ×2**).
2. **Settings → Internet → On.** This needs a phone-verified Kaggle
   account; without internet the notebook cannot download the weights or
   publish a link.

## How to use it

1. Run the cells in order. The first run installs the model and
   downloads about 2 GB of weights — give it 5–10 minutes.
2. The last cell prints `Running on public URL: https://....gradio.live`
3. On the machine running the backend, set that whole link:

   ```
   python -m backend.set_secret TRYON_FALLBACK_URL
   ```

   and make sure `.env` has
   `TRYON_PROVIDERS=self_hosted,self_hosted_2,fashn_space`
   so your own notebook is tried first and the public Space is only the
   last resort.
4. Keep this tab open. When the link changes (every restart), set it
   again.

If you run the Colab notebook as well, put its link in
`TRYON_FALLBACK_URL_2` — two notebooks are two separate free GPUs, so
one going to sleep is no longer an outage.


## 1. Check the GPU

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if result.returncode != 0:
    raise SystemExit(
        "No GPU in this session.\n\n"
        "Settings -> Accelerator -> GPU P100 (or T4 x2), then run this "
        "cell again. On CPU one try-on takes many minutes, which is not "
        "usable."
    )

print(result.stdout)
print("GPU found - carry on.")


## 2. Check internet access

Kaggle disables outbound network by default. Without it the weights
cannot be downloaded and no public link can be published, so this is
worth failing on now rather than three cells later.


In [ ]:
import socket

try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError:
    raise SystemExit(
        "No internet access in this session.\n\n"
        "Settings -> Internet -> On, then run this cell again. Kaggle "
        "requires a phone-verified account to enable it."
    )

print("internet is on - carry on.")


## 3. Install the model

FASHN VTON v1.5, **Apache-2.0** — chosen over IDM-VTON, CatVTON,
OOTDiffusion, StableVITON and VITON-HD because those are CC BY-NC-SA
4.0 and forbid commercial use.


In [ ]:
!pip install -q "git+https://github.com/fashn-AI/fashn-vton-1.5.git" gradio huggingface_hub

print("installed")


## 4. Download the weights and load the model

About 2 GB. Kaggle's working directory is wiped between sessions, so
this runs again each time.


In [ ]:
from huggingface_hub import snapshot_download

WEIGHTS_DIR = "/kaggle/working/weights"

snapshot_download(repo_id="fashn-ai/fashn-vton-1.5", local_dir=WEIGHTS_DIR)

from fashn_vton import TryOnPipeline

pipeline = TryOnPipeline(weights_dir=WEIGHTS_DIR)

print("model loaded")


## 5. Warm the model up (optional but worth it)

The first real request is always the slowest, because CUDA kernels
compile on first use. Doing that here on a blank image means the first
user does not wait for it — and does not time out because of it.


In [ ]:
from PIL import Image

try:
    blank_person = Image.new("RGB", (768, 1024), (128, 128, 128))
    blank_garment = Image.new("RGB", (768, 1024), (200, 200, 200))
    import time
    started = time.time()
    pipeline(person_image=blank_person, garment_image=blank_garment,
             category="tops")
    print(f"warm-up pass took {time.time() - started:.1f}s "
          f"(real requests should be faster)")
except Exception as error:
    # Not fatal - a grey rectangle is not a person, so the model is
    # entitled to refuse it. The kernels are compiled either way.
    print(f"warm-up finished with {type(error).__name__} - that is fine")


## 6. Start the service

Prints the public address. Copy the whole `https://....gradio.live` line.

The endpoint is named `try_on` and takes three inputs **in this order**:
person image, garment image, category (`tops`, `bottoms` or
`one-pieces`). That contract is what `backend/virtual_tryon.py` expects —
changing it breaks the backend.

`max_size=1` on the queue is deliberate: one generation at a time. Two
at once on a 16 GB card risks running out of memory halfway through,
which fails BOTH of them. The backend already allows only one try-on per
account at a time, and it waits politely.


In [ ]:
import gradio as gr

CATEGORIES = ["tops", "bottoms", "one-pieces"]


def try_on(person_image, garment_image, category):
    """One garment onto one person. The backend chains calls for a full
    outfit - bottom first, then the top applied to that result."""
    if person_image is None:
        raise gr.Error("No photo of the person was supplied.")

    if garment_image is None:
        raise gr.Error("No garment image was supplied.")

    if category not in CATEGORIES:
        raise gr.Error(
            f"Category must be one of {', '.join(CATEGORIES)} - got "
            f"'{category}'."
        )

    result = pipeline(
        person_image=person_image.convert("RGB"),
        garment_image=garment_image.convert("RGB"),
        category=category,
    )

    return result.images[0]


with gr.Blocks(title="Wardrobe-AI Try-On") as demo:

    gr.Markdown("## Wardrobe-AI virtual try-on (Kaggle free GPU)")

    with gr.Row():
        person = gr.Image(type="pil", label="Person")
        garment = gr.Image(type="pil", label="Garment")

    category = gr.Dropdown(CATEGORIES, value="tops", label="Category")

    output = gr.Image(type="pil", label="Result")

    gr.Button("Try on", variant="primary").click(
        try_on,
        inputs=[person, garment, category],
        outputs=output,
        api_name="try_on",
    )

# One generation at a time: two on a 16 GB card can run out of memory
# and fail both. show_error surfaces the real reason to the backend
# instead of a blank 500.
demo.queue(max_size=1).launch(share=True, show_error=True, quiet=False)


## Keeping it alive

* The session ends after at most **12 hours**, and sooner if the tab is
  closed for a long stretch or Kaggle needs the GPU back.
* The `gradio.live` link **changes every restart**, so it has to be set
  in `.env` again each time.
* Weekly GPU hours are shown on your Kaggle account page. Thirty hours
  is generous for ten try-ons a day per user, but it is finite: leaving
  this running overnight for no reason spends it.
* `gradio.live` links are **public while they last**. Anyone with the
  link can send images to your notebook. Do not post it anywhere, and
  restart the notebook if you ever paste it somewhere by accident.
